# Tutorial - Benchmark

How expensive are the simulations? This notebook times one optical sequence (laser $\to$ free $\to$ MW) for each model and execution mode:

- **7-level** (no-HF) vs **14-level** (Doherty / Duarte, NV $\otimes\ ^{15}$N) — the Hilbert-space size roughly doubles, so the master-equation solve gets noticeably slower.
- **serial** vs **parallel** — the parallel runs use `qt.loky_pmap` to spread the repetitions across CPU cores.

Timing uses `time.perf_counter` over several repetitions; the bars show the mean with $\pm$ std error bars.

`RECOMPUTE = True` runs the full benchmark (`runs=8, repeats=5`); `False` loads the shipped reduced cache. **Timings are machine-dependent** — recompute on your own hardware for meaningful numbers.

In [ ]:
RECOMPUTE = True
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from examples import protocol_runs as engine

In [ ]:
data = engine.run_or_load(
    "benchmark",
    lambda: engine.compute_benchmark(["no-HF", "Doherty", "Duarte"], runs=8, repeats=5),
    recompute=RECOMPUTE,
)
engine.plot_benchmark_result(data)
for label, r in data["results"].items():
    print(f"{label:8s} serial={r['seq_mean']:.3f}±{r['seq_std']:.3f}s  "
          f"parallel={r['par_mean']:.3f}±{r['par_std']:.3f}s")
plt.show()

In [ ]:
# --- Smoke test: tiny self-contained run to confirm this notebook works ---
try:
    engine.plot_benchmark_result(
        engine.compute_benchmark(["no-HF"], runs=1, repeats=1), save_as=None)
    plt.close("all")
    print("SMOKE OK: benchmark")
except Exception as _e:
    print("SMOKE FAIL:", repr(_e))